# OCR ingestion lab — `bctc_ingest`

Turn a Vietnamese financial-statement PDF into **retrieval chunks saved as JSON**.

This notebook runs **offline**: it generates a tiny sample PDF so every cell works without
the real corpus or a network connection. To run on real data, point the paths at a pack
built by the `data-pipeline` component (`corpus/` + `catalog.csv`).

Pipeline per page: **route** (text vs OCR) → **extract** (PyMuPDF / pypdf) or **OCR**
(PaddleOCR, CPU) → **clean** → **chunk** → **JSON**.

## 0. Setup

Recommended: install the package into **this kernel's** environment with `%pip install -e ..`
(run from `ocr-pipeline/`, add `..[ocr]` for the PaddleOCR extra). The cell below also works
**without installing**: if `bctc_ingest` is not importable it adds the `src/` folder to the
path by searching upward for it — so the notebook runs from a fresh checkout either way.

In [ ]:
# !pip install paddlepaddle

In [1]:
# !pip install paddleocr

In [2]:
# Uncomment to install into this kernel instead of using the sys.path fallback below:
# %pip install -e ..
import sys, pathlib
try:
    import bctc_ingest  # noqa: F401
except ModuleNotFoundError:
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / 'src' / 'bctc_ingest').is_dir():
            sys.path.insert(0, str(base / 'src'))
            print('added to sys.path:', base / 'src')
            break
    import bctc_ingest  # noqa: F401

import json, tempfile
from pathlib import Path
import pymupdf  # PyMuPDF

from bctc_ingest import route, clean, chunk as chunker
from bctc_ingest.extract import PdfDoc
from bctc_ingest.pipeline import IngestConfig, ingest_document, export_jsonl
from bctc_ingest import ocr_paddle

print('PaddleOCR available:', ocr_paddle.available())  # False is fine — text path still works

C:\Users\ATBK\.conda\envs\pickleball\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PaddleOCR available: True


## 1. A sample document

We build a 3-page text PDF so the notebook is self-contained. Replace `PDF` with a path
into your real corpus to try the actual reports.

In [3]:
work = Path(tempfile.mkdtemp())
doc = pymupdf.open()
body = ('Bao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. '
        'Tong cong tai san va nguon von duoc trinh bay day du theo Thong tu 200. ') * 8
for i in range(3):
    page = doc.new_page()
    page.insert_text((72, 72), f'Trang {i+1}\n\n{body}\n\nMa so 270 Tong cong tai san.', fontsize=11)
PDF = work / 'VNM_BCTC_HN_N2025.pdf'
doc.save(PDF); doc.close()
print('sample pdf:', PDF)

sample pdf: C:\Users\ATBK\AppData\Local\Temp\tmpjmti7vs5\VNM_BCTC_HN_N2025.pdf


## 2. Routing — does a page need OCR?

A page counts as usable text only if it has enough characters **and** enough Vietnamese
diacritics (a scanned filing often has a thin accent-less text layer that fools a plain
character count). Pages that fail are sent to OCR.

In [4]:
with PdfDoc(PDF) as d:
    page_texts = d.page_texts()
methods = route.route_pages(page_texts)
print('per-page methods:', methods)
print('document method:', route.doc_method(methods))
for i, t in enumerate(page_texts):
    print(f'  page {i+1}: chars={len(t):4d}  diacritic_ratio={route.diacritic_ratio(t):.2f}')

per-page methods: ['ocr', 'ocr', 'ocr']
document method: ocr
  page 1: chars= 144  diacritic_ratio=0.00
  page 2: chars= 144  diacritic_ratio=0.00
  page 3: chars= 144  diacritic_ratio=0.00


## 3. Text extraction

For a text page we take the embedded layer directly (PyMuPDF, with a pypdf fallback).

In [5]:
print(page_texts[0][:300])

Trang 1
Bao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay
Ma so 270 Tong cong tai san.



## 4. The OCR path (PaddleOCR)

For a scanned page we render it to an image and run PaddleOCR with the small CPU models.
PaddleOCR is an optional dependency — enable it with `pip install -e ".[ocr]"`. The code
below only runs if it is installed, so this notebook never fails without it.

In [7]:
# PDF là file pdf path --> điền file pdf thực tế ví dụ mã VNM --> để xem output

In [6]:
if ocr_paddle.available():
    engine = ocr_paddle.PaddleOCREngine(lang='vi', dpi=200)
    with PdfDoc(PDF) as d:
        text, conf = engine.ocr_page(d, 0)   # OCR page 1
    print('OCR confidence:', conf)
    print(text[:300])
else:
    print('PaddleOCR not installed — skipping. Install:  pip install -e ".[ocr]"')

C:\Users\ATBK\.conda\envs\pickleball\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_medium_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `C:\Users\ATBK\.paddlex\official_models\PP-OCRv6_medium_det`.
Fetching 5 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.27it/s]
Creating model: ('PP-OC

OCR confidence: 0.9883
Trang 1
Bao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay
Ma so 270 Tong cong tai san.


## 5. Cleaning

Normalise Unicode (NFC) and whitespace, and drop lines that repeat on most pages
(headers, footers, the digital-signature stamp).

In [8]:
cleaned = clean.strip_furniture([clean.normalize(t) for t in page_texts])
print(cleaned[0][:300])

Trang 1
Bao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay
Ma so 270 Tong cong tai san.


## 6. Chunking

Pack paragraphs into ~`chunk_size`-character chunks with `overlap` characters of context
carried between them; each chunk remembers the pages it spans.

In [9]:
pages = [(i+1, t) for i, t in enumerate(cleaned) if t.strip()]
chunks = chunker.chunk_pages(pages, chunk_size=500, overlap=80)
print('n_chunks:', len(chunks))
print('chunk 0 pages:', chunks[0]['pages'], '| n_chars:', chunks[0]['n_chars'])
print(chunks[0]['text'][:200])

n_chunks: 1
chunk 0 pages: [1, 2, 3] | n_chars: 433
Trang 1
Bao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay
Ma so 270 Tong cong tai san.

Trang 2
Bao cao tai chinh hop nhat nam 2025. Don vi tin


## 7. The full pipeline → one JSON per document

`ingest_document` does routing → extract/OCR → clean → chunk → write
`<out>/chunks/<TICKER>/<doc_id>.json`, carrying the document metadata onto every chunk.

In [10]:
meta = dict(ticker='VNM', year=2025, quarter=0, period='2025', scope='hopnhat',
            audited=True, unit='trieu', exchange='HOSE')
cfg = IngestConfig(chunk_size=500, overlap=80)
res = ingest_document(PDF, work/'out', cfg, meta=meta, ocr_engine=None, force='text')
print(res)

doc_json = json.loads((work/'out'/res['json_path']).read_text(encoding='utf-8'))
print('\n--- first chunk ---')
print(json.dumps(doc_json['chunks'][0], ensure_ascii=False, indent=2)[:600])

{'doc_id': 'VNM_BCTC_HN_N2025', 'ticker': 'VNM', 'source_file': 'VNM_BCTC_HN_N2025.pdf', 'status': 'ok', 'extract_method': 'text', 'n_pages': 3, 'n_ocr_pages': 0, 'n_chunks': 1, 'mean_ocr_conf': '', 'json_path': 'chunks\\VNM\\VNM_BCTC_HN_N2025.json'}

--- first chunk ---
{
  "chunk_id": "VNM_BCTC_HN_N2025::0000",
  "doc_id": "VNM_BCTC_HN_N2025",
  "text": "Trang 1\nBao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay\nMa so 270 Tong cong tai san.\n\nTrang 2\nBao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay\nMa so 270 Tong cong tai san.\n\nTrang 3\nBao cao tai chinh hop nhat nam 2025. Don vi tinh trieu dong. Tong cong tai san va nguon von duoc trinh bay\nMa so 270 Tong cong tai san.",
  "pages": [
    1,
    2,
    3
  ],
  "extract_method": "text",
 


## 8. Running on the whole corpus

In practice you run the CLI over a corpus the `data-pipeline` built. From `ocr-pipeline/`:

```bash
python -m bctc_ingest run \
    --corpus ../data-pipeline/pack_real/corpus \
    --catalog ../data-pipeline/pack_real/catalog.csv \
    --out ingest_out
python -m bctc_ingest export --out ingest_out   # -> ingest_out/chunks_all.jsonl
```

It is resumable (a `manifest.csv` records what is done) and skips finished documents on a
re-run. Use `--force-ocr` to OCR everything, `--chunk-size` / `--overlap` to tune chunks.

In [11]:
# programmatic equivalent for a folder of PDFs:
# from bctc_ingest.pipeline import ingest_corpus
# ingest_corpus('../data-pipeline/pack_real/corpus', 'ingest_out', cfg,
#               catalog_path='../data-pipeline/pack_real/catalog.csv')
export_jsonl(work/'out')   # demo the export on our one-document output
print(sorted(p.name for p in (work/'out').rglob('*')) )

export: 1 chunks -> C:\Users\ATBK\AppData\Local\Temp\tmpjmti7vs5\out\chunks_all.jsonl
['VNM', 'VNM_BCTC_HN_N2025.json', 'chunks', 'chunks_all.jsonl']


## 9. Storage layout & what comes next

```
ingest_out/
├── chunks/
│   ├── VNM/VNM_BCTC_HN_N2025.json     # one JSON per document (report)
│   └── FPT/...
├── manifest.csv                        # index: doc, method, #chunks, status
└── chunks_all.jsonl                    # optional flat export for bulk embedding
```

**One JSON per document** (not one giant file, not one file per chunk): it matches the
crawler's unit of work, keeps runs resumable and parallelisable, and stays easy to inspect.

**Next step — the database.** These JSON chunks are the input to the *embedding + indexing*
component: each chunk's `text` is embedded and the vector + metadata are upserted into a
**vector database** (e.g. Qdrant / pgvector / FAISS). The JSON stays as the durable,
inspectable intermediate, so you can re-embed with a new model without re-running OCR. See
`docs/STORAGE.md` for the full rationale.